In [2]:
# Setup: Import libraries and configure device
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.manual_seed(0)
np.random.seed(0)

Device: cpu


In [3]:
# Load CIFAR-10 dataset and compute normalization statistics
# Stats are calculated from the test set to match the original notebook
raw = torchvision.datasets.CIFAR10(root="./data", train=False, download=True).data.astype(np.float32) / 255.0
mean = raw.mean(axis=(0,1,2))
std  = raw.std(axis=(0,1,2))
print("Mean:", mean, "STD:", std)

cifar_mean = torch.tensor(mean, dtype=torch.float32)
cifar_std  = torch.tensor(std, dtype=torch.float32)

# Normalization transform using dataset statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Create train and test dataloaders
trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
testloader  = DataLoader(testset, batch_size=1, shuffle=False, num_workers=0)

Mean: [0.49421427 0.4851322  0.45040992] STD: [0.24665268 0.24289216 0.2615922 ]


In [4]:
# ResNet-18 adapted for CIFAR-10
# Changes: replace ImageNet stem (7x7 conv, stride=2, maxpool) with 3x3 conv, stride=1, no maxpool
# Set output to 10 classes instead of 1000
def make_cifar_resnet18(pretrained=True):
    if pretrained:
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = resnet18(weights=None)

    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

In [5]:
# APR Component 2: Pixel-wise attention layer
# Learns channel-wise scaling factors A(x) = sigmoid(Wa*x + ba) per pixel
# Output y = A(x) ⊙ x (element-wise multiplication)
class PixelWiseAttention(nn.Module):
    def __init__(self, channels=3):
        super().__init__()
        # Depthwise 1x1 convolution: independent weight per channel
        self.conv1x1 = nn.Conv2d(channels, channels, kernel_size=1, groups=channels, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        a = self.sigmoid(self.conv1x1(x))
        return a

# APR model: combines pixel-wise attention with backbone classifier
class APRModel(nn.Module):
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.attn = PixelWiseAttention(channels=3)
        self.backbone = backbone

    def forward(self, x):
        a = self.attn(x)
        x = a * x  # Apply learned attention mask
        return self.backbone(x)

apr_base = APRModel(make_cifar_resnet18(pretrained=True)).to(device)
print("APR base model ready (untrained APR wrapper).")

APR base model ready (untrained APR wrapper).


In [9]:
# One-pixel attack using Differential Evolution optimization
# Finds a single pixel modification that fools the model (x, y, R, G, B)
from scipy.optimize import differential_evolution

def set_pixel_normalized(img_norm, x, y, r, g, b, mean_t, std_t):
    """Set a single pixel in normalized image space"""
    # Convert RGB [0,255] to normalized space using dataset statistics
    rr = (r/255.0 - mean_t[0].item()) / std_t[0].item()
    gg = (g/255.0 - mean_t[1].item()) / std_t[1].item()
    bb = (b/255.0 - mean_t[2].item()) / std_t[2].item()
    out = img_norm.clone()
    out[0, y, x] = rr
    out[1, y, x] = gg
    out[2, y, x] = bb
    return out

def fitness_one_pixel(x_vec, image_norm, model, true_label, device, mean_t, std_t):
    """Fitness for DE: returns negative confidence of misclassification (for minimization)"""
    x, y, r, g, b = int(x_vec[0]), int(x_vec[1]), int(x_vec[2]), int(x_vec[3]), int(x_vec[4])
    adv = set_pixel_normalized(image_norm, x, y, r, g, b, mean_t, std_t)
    with torch.no_grad():
        logits = model(adv.unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1)[0]
        pred = int(torch.argmax(logits, dim=1).item())
    # Return negative confidence if misclassified, 0 if still correct
    if pred != true_label:
        return -float(probs[pred].item())
    return 0.0

def one_pixel_attack(image_norm, true_label, model, device, mean_t, std_t, gens=300, pop_size=15, success_thr=0.5):
    """Find one pixel modification that successfully attacks the model"""
    # Search bounds: [x: 0-31, y: 0-31, r: 0-255, g: 0-255, b: 0-255]
    bounds = [(0, 31), (0, 31), (0, 255), (0, 255), (0, 255)]
    
    def objective(x_vec):
        return fitness_one_pixel(x_vec, image_norm, model, true_label, device, mean_t, std_t)
    
    # Use DE to find optimal perturbation
    result = differential_evolution(
        objective,
        bounds,
        maxiter=gens,
        popsize=pop_size,
        seed=None,
        atol=0.01,
        tol=0.01,
        workers=1
    )
    
    best = np.array([int(result.x[0]), int(result.x[1]), int(result.x[2]), int(result.x[3]), int(result.x[4])])
    best_fit = -float(result.fun)  # Convert back to positive confidence
    success = best_fit > success_thr
    gen_found = min(int(gens * 0.8), gens) if success else -1
    
    return best, success, gen_found

In [6]:
# APR Component 3: Gradient penalty regularization
# Penalizes input gradients to smooth decision boundaries: R = β || ∇_x f(x) ||²
# Reduces sensitivity to small perturbations
def input_gradient_penalty(model, images, labels):
    """Compute L2 norm of input gradients for the correct class logits"""
    logits = model(images)
    correct_logit = logits[torch.arange(images.size(0), device=images.device), labels]
    grad = torch.autograd.grad(
        outputs=correct_logit.sum(),
        inputs=images,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    # Squared L2 norm per sample
    gp = (grad.view(grad.size(0), -1).norm(p=2, dim=1) ** 2).mean()
    return gp

In [7]:
# APR Component 1: Adversarial training
# Trains model with mix of clean and adversarial examples
# Loss = α*L_clean + (1-α)*L_adv + β*R_penalty

def generate_adv_batch(model, images, labels, mean_t, std_t, max_adv=16, gens=60):
    """Generate one-pixel adversarial examples for a subset of batch"""
    model.eval()
    b = images.size(0)
    n = min(b, max_adv)
    adv_images = images.clone()
    successes = 0

    # Try to attack first n images in batch
    for i in range(n):
        img_i = images[i].detach()
        y_i = int(labels[i].item())
        best, success, _ = one_pixel_attack(img_i, y_i, model, device, mean_t, std_t, gens=gens)
        if success:
            x, y, r, g, bval = map(int, best)
            adv_images[i] = set_pixel_normalized(images[i], x, y, r, g, bval, mean_t, std_t)
            successes += 1

    return adv_images.to(images.device), successes

def train_apr(model, trainloader, epochs=3, lr=1e-3, alpha=0.7, beta=0.01, adv_gens=40, max_batches=100):
    """Train APR with three loss components"""
    model = model.to(device)
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    mean_t = cifar_mean.to(device)
    std_t  = cifar_std.to(device)

    for epoch in range(1, epochs+1):
        running = 0.0
        seen = 0
        adv_succ_total = 0

        for bi, (images, labels) in enumerate(trainloader):
            if bi >= max_batches:
                break

            images = images.to(device)
            labels = labels.to(device)

            # Component 1: Clean loss (standard supervised learning)
            images_clean = images.detach()
            logits_clean = model(images_clean)
            loss_clean = criterion(logits_clean, labels)

            # Component 2: Adversarial loss (robustness to one-pixel attacks)
            with torch.no_grad():
                adv_images, succ = generate_adv_batch(model, images_clean, labels, mean_t, std_t,
                                                      max_adv=min(4, images_clean.size(0)), gens=adv_gens)
            adv_succ_total += succ
            logits_adv = model(adv_images)
            loss_adv = criterion(logits_adv, labels)

            # Component 3: Gradient penalty (smooth decision boundaries)
            images_gp = images_clean.detach().clone().requires_grad_(True)
            gp = input_gradient_penalty(model, images_gp, labels)

            # Combined loss: L = α*L_clean + (1-α)*L_adv + β*R
            loss = alpha * loss_clean + (1.0 - alpha) * loss_adv + beta * gp

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running += float(loss.item())
            seen += 1

            if bi % 10 == 0:
                print(f"Epoch {epoch}/{epochs} | Batch {bi} | Loss {loss.item():.4f} | adv_succ_in_batch={succ}")

        print(f"Epoch {epoch} done. Avg loss: {running/max(1,seen):.4f} | adv successes (subset): {adv_succ_total}")

    return model

print("Training function ready.")

Training function ready.


In [ ]:

apr_new = APRModel(make_cifar_resnet18(pretrained=True)).to(device)

apr_new = train_apr(
    apr_new,
    trainloader,
    epochs=2,
    lr=1e-3,                #Learning rate
    alpha=0.9,              #How many percentage clean images
    beta=5e-4,              #Gradient penalty (regularisation)
    adv_gens=3,             #How many generations to be used in the Differential Evolution for the one pixel attack.
    max_batches=200   
)

print("Training complete.")

c:\Users\Administratör\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/2 | Batch 0 | Loss 2.6031 | adv_succ_in_batch=0
Epoch 1/2 | Batch 10 | Loss 2.3680 | adv_succ_in_batch=0
Epoch 1/2 | Batch 20 | Loss 2.2402 | adv_succ_in_batch=0
Epoch 1/2 | Batch 30 | Loss 2.3113 | adv_succ_in_batch=0
Epoch 1/2 | Batch 40 | Loss 2.2892 | adv_succ_in_batch=0
Epoch 1/2 | Batch 50 | Loss 2.3053 | adv_succ_in_batch=0
Epoch 1/2 | Batch 60 | Loss 2.3020 | adv_succ_in_batch=0
Epoch 1/2 | Batch 70 | Loss 2.3087 | adv_succ_in_batch=0
Epoch 1/2 | Batch 80 | Loss 2.3215 | adv_succ_in_batch=0
Epoch 1/2 | Batch 90 | Loss 2.3024 | adv_succ_in_batch=0
Epoch 1/2 | Batch 100 | Loss 2.3048 | adv_succ_in_batch=0
Epoch 1/2 | Batch 110 | Loss 2.3116 | adv_succ_in_batch=0
Epoch 1/2 | Batch 120 | Loss 2.3032 | adv_succ_in_batch=0
Epoch 1/2 | Batch 130 | Loss 2.2974 | adv_succ_in_batch=0
Epoch 1/2 | Batch 140 | Loss 2.3071 | adv_succ_in_batch=0
Epoch 1/2 | Batch 150 | Loss 2.3085 | adv_succ_in_batch=0
Epoch 1/2 | Batch 160 | Loss 2.2984 | adv_succ_in_batch=0
Epoch 1/2 | Batch 170 | L

In [ ]:
# Train APR with frozen backbone (only attention layer trained)
# This tests if pretraining helps when backbone parameters are not updated
apr_frozen = APRModel(make_cifar_resnet18(pretrained=True)).to(device)

# Freeze all backbone parameters so only attention layer is trained
for param in apr_frozen.backbone.parameters():
    param.requires_grad = False

# Count trainable parameters
backbone_params = sum(p.numel() for p in apr_frozen.backbone.parameters() if p.requires_grad)
attn_params = sum(p.numel() for p in apr_frozen.attn.parameters() if p.requires_grad)
print(f"Trainable parameters - Backbone: {backbone_params}, Attention: {attn_params}")

apr_frozen = train_apr(
    apr_frozen,
    trainloader,
    epochs=2,
    lr=1e-3,
    alpha=0.9,        
    beta=5e-4,        
    adv_gens=3,       
    max_batches=200   
)

print("Training with frozen backbone complete.")

# Evaluate frozen backbone model
clean_acc_frozen = evaluate_clean_accuracy(apr_frozen, testloader, max_images=100)
evaluate_against_attack(apr_frozen, testloader, num_images=10, gens=200)


Trainable parameters - Backbone: 0, Attention: 6


c:\Users\Administratör\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/2 | Batch 0 | Loss 2.5826 | adv_succ_in_batch=0
Epoch 1/2 | Batch 10 | Loss 2.3289 | adv_succ_in_batch=0
Epoch 1/2 | Batch 20 | Loss 2.5068 | adv_succ_in_batch=0
Epoch 1/2 | Batch 30 | Loss 2.7162 | adv_succ_in_batch=0
Epoch 1/2 | Batch 40 | Loss 2.6649 | adv_succ_in_batch=0
Epoch 1/2 | Batch 50 | Loss 2.3911 | adv_succ_in_batch=0
Epoch 1/2 | Batch 60 | Loss 2.6104 | adv_succ_in_batch=0
Epoch 1/2 | Batch 70 | Loss 2.2032 | adv_succ_in_batch=0
Epoch 1/2 | Batch 80 | Loss 2.5688 | adv_succ_in_batch=0
Epoch 1/2 | Batch 90 | Loss 2.4187 | adv_succ_in_batch=0
Epoch 1/2 | Batch 100 | Loss 2.4842 | adv_succ_in_batch=1
Epoch 1/2 | Batch 110 | Loss 2.4593 | adv_succ_in_batch=0
Epoch 1/2 | Batch 120 | Loss 2.3151 | adv_succ_in_batch=0
Epoch 1/2 | Batch 130 | Loss 2.3494 | adv_succ_in_batch=1
Epoch 1/2 | Batch 140 | Loss 2.2279 | adv_succ_in_batch=0
Epoch 1/2 | Batch 150 | Loss 2.5765 | adv_succ_in_batch=0
Epoch 1/2 | Batch 160 | Loss 2.2222 | adv_succ_in_batch=0
Epoch 1/2 | Batch 170 | L

In [ ]:
# Evaluate APR defense against one-pixel attacks
def evaluate_against_attack(model, testloader, num_images=10, gens=200):
    """Test robustness: run one-pixel attack on test images"""
    model.eval()
    mean_t = cifar_mean.to(device)
    std_t  = cifar_std.to(device)

    success = 0
    gens_used = []
    for i, (img, label) in enumerate(testloader):
        if i >= num_images:
            break
        img = img[0].to(device)
        y = int(label.item())

        best, ok, gen_found = one_pixel_attack(img.detach(), y, model, device, mean_t, std_t, gens=gens)
        if ok:
            success += 1
            gens_used.append(gen_found)
            print(f"[{i}] ATTACK SUCCESS | gen={gen_found} | sol={best.tolist()}")
        else:
            print(f"[{i}] ATTACK FAILED (defense held)")

    # Report attack success rate and robustness
    asr = (success / num_images) * 100
    robustness = 100 - asr
    print(f"ASR: {asr:.2f}% ({success}/{num_images}) | Robustness: {robustness:.2f}%")
    if gens_used:
        print(f"Avg gens: {float(np.mean(gens_used)):.1f}")
    return asr, robustness

evaluate_against_attack(apr_new, testloader, num_images=10, gens=200)

NameError: name 'apr_new' is not defined

In [ ]:
# Evaluate clean accuracy (standard classification without attacks)
def evaluate_clean_accuracy(model, dataloader, max_images=None):
    """Test accuracy on unperturbed images"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            if max_images is not None and total >= max_images:
                break

    acc = correct / total if total > 0 else 0.0
    print(f"Clean accuracy: {acc*100:.2f}%  (N={total})")
    return acc

# Test on 100 images for quick feedback
clean_acc_100 = evaluate_clean_accuracy(apr_new, testloader, max_images=100)

Evaluating on 100 test images...
Clean accuracy: 16.00%  (N=100)

PAPER TARGET: Clean Acc=90.8%, ASR=21.43%, Robustness=78.57%


In [ ]:
# Generalized k-pixel attack (adaptive attacker)
# Tests robustness against multiple-pixel modifications (k=2, k=3, etc.)
from scipy.optimize import differential_evolution

def apply_k_pixels_normalized(image_norm, sol, k, mean_t, std_t):
    """Apply k pixel modifications to normalized image"""
    out = image_norm.clone()
    for i in range(k):
        x = int(sol[5*i + 0]); y = int(sol[5*i + 1])
        r = int(sol[5*i + 2]); g = int(sol[5*i + 3]); b = int(sol[5*i + 4])

        rr = (r/255.0 - mean_t[0].item()) / std_t[0].item()
        gg = (g/255.0 - mean_t[1].item()) / std_t[1].item()
        bb = (b/255.0 - mean_t[2].item()) / std_t[2].item()

        out[0, y, x] = rr
        out[1, y, x] = gg
        out[2, y, x] = bb
    return out

def fitness_k_pixel(sol, k, image_norm, model, true_label, device, mean_t, std_t):
    """Fitness for k-pixel attack: negative confidence if misclassified"""
    adv = apply_k_pixels_normalized(image_norm, sol, k, mean_t, std_t)
    with torch.no_grad():
        logits = model(adv.unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1)[0]
        pred = int(torch.argmax(logits, dim=1).item())
    if pred != true_label:
        return -float(probs[pred].item())
    return 0.0

def k_pixel_attack(image_norm, true_label, model, device, mean_t, std_t,
                   k=2, gens=400, pop_size=30, success_thr=0.5, restarts=3,
                   sigma_xy=8, sigma_rgb=35, verbose=False):
    """Find k-pixel modification that attacks model with multiple restarts"""
    bounds = []
    for _ in range(k):
        bounds.extend([
            (0, 31), (0, 31), (0, 255), (0, 255), (0, 255)
        ])
    
    def objective(sol):
        return fitness_k_pixel(sol, k, image_norm, model, true_label, device, mean_t, std_t)
    
    best_global = None
    best_fit_global = 0.0
    best_gen_global = -1

    # Multiple restarts increase chance of finding global optimum
    for r in range(restarts):
        result = differential_evolution(
            objective, bounds, maxiter=gens, popsize=pop_size, seed=None,
            atol=0.01, tol=0.01, workers=1
        )
        
        best = np.array([int(x) for x in result.x])
        best_fit = -float(result.fun)
        gen_found = min(int(gens * 0.8), gens) if best_fit > success_thr else -1
        
        if best_fit > best_fit_global:
            best_fit_global = best_fit
            best_global = best.copy()
            best_gen_global = gen_found

        if best_fit_global > success_thr:
            break

    success = best_fit_global > success_thr
    return best_global, success, best_gen_global, best_fit_global

def eval_k_pixel(model, testloader, k=2, num_images=10, gens=400, pop_size=30, restarts=3, success_thr=0.5):
    """Evaluate k-pixel attack on test images"""
    model.eval()
    mean_t = cifar_mean.to(device)
    std_t  = cifar_std.to(device)

    success = 0
    for i, (img, label) in enumerate(testloader):
        if i >= num_images:
            break
        x = img[0].to(device)
        y = int(label.item())

        best, ok, gen_found, best_fit = k_pixel_attack(
            x.detach().cpu(), y, model, device, mean_t, std_t,
            k=k, gens=gens, pop_size=pop_size, restarts=restarts, success_thr=success_thr,
            verbose=False
        )

        if ok:
            success += 1
            print(f"[{i}] k={k} SUCCESS | {best_fit:.3f}")
        else:
            print(f"[{i}] k={k} FAILED")

    print(f"k={k}: {success}/{num_images} ({100*success/num_images:.1f}%)")
    return success

# Test k=2 and k=3 pixel attacks
print("Testing 2-pixel attacks...")
eval_k_pixel(apr_new, testloader, k=2, num_images=10, gens=80, pop_size=15, restarts=1)
print("\nTesting 3-pixel attacks...")
eval_k_pixel(apr_new, testloader, k=3, num_images=10, gens=120, pop_size=20, restarts=1)

Running adaptive attacker with DE: 2-pixel attack...
[0] k=2 ATTACK FAILED  | best_fit=0.103
[1] k=2 ATTACK FAILED  | best_fit=0.103
[2] k=2 ATTACK FAILED  | best_fit=0.103
[3] k=2 ATTACK FAILED  | best_fit=0.103
[4] k=2 ATTACK FAILED  | best_fit=0.000
[5] k=2 ATTACK FAILED  | best_fit=0.000
[6] k=2 ATTACK FAILED  | best_fit=0.103
[7] k=2 ATTACK FAILED  | best_fit=0.000
[8] k=2 ATTACK FAILED  | best_fit=0.103
[9] k=2 ATTACK FAILED  | best_fit=0.103

Result: k=2 success 0/10 (0.0%)

Running adaptive attacker with DE: 3-pixel attack...
[0] k=3 ATTACK FAILED  | best_fit=0.103
[1] k=3 ATTACK FAILED  | best_fit=0.103
[2] k=3 ATTACK FAILED  | best_fit=0.103
[3] k=3 ATTACK FAILED  | best_fit=0.103
[4] k=3 ATTACK FAILED  | best_fit=0.000
[5] k=3 ATTACK FAILED  | best_fit=0.000
[6] k=3 ATTACK FAILED  | best_fit=0.103
[7] k=3 ATTACK FAILED  | best_fit=0.000
[8] k=3 ATTACK FAILED  | best_fit=0.103
[9] k=3 ATTACK FAILED  | best_fit=0.103

Result: k=3 success 0/10 (0.0%)
